# Prepare Article Records

This notebook reads the canonical nested legal JSON files and creates article-level JSONL records for BM25 and semantic search. It does not tokenize, segment, embed, or modify the original JSON.

In [1]:
import json
from collections import Counter
from pathlib import Path

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'react-client').is_dir() and (path / 'data-pipeline').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
SOURCE_DIR = DATA_DIR / 'processed' / 'json'
OUTPUT_PATH = DATA_DIR / 'processed' / 'search' / 'articles.jsonl'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Input:', SOURCE_DIR)
print('Output:', OUTPUT_PATH)

Input: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/json
Output: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/search/articles.jsonl


In [2]:
def article_record(document, article, chapter=None, section=None):
    document_id = str(document['document_id'])
    article_number = article.get('article_number')
    if article_number is None:
        raise ValueError(f'Article number is missing in document {document_id}')
    article_id = f'{document_id}-{article_number}'
    record = {
        'id': article_id,
        'article_id': article_id,
        'document_id': document_id,
        'article_number': article_number,
        'text': article['text'],
        'document_title': document.get('title', ''),
        'document_number': document.get('document_number'),
        'document_type': document.get('document_type'),
        'source_url': document.get('detail_url') or document.get('pdf_url'),
        'pdf_url': document.get('pdf_url'),
    }
    if chapter:
        record['chapter_number'] = chapter.get('chapter_number')
        record['chapter_title'] = chapter.get('chapter_title', '')
    if section:
        record['section_number'] = section.get('section_number')
        record['section_title'] = section.get('section_title', '')
    return record


def iter_articles(document):
    for article in document.get('articles', []):
        yield article_record(document, article)
    for chapter in document.get('chapters', []):
        for article in chapter.get('articles', []):
            yield article_record(document, article, chapter=chapter)
        for section in chapter.get('sections', []):
            for article in section.get('articles', []):
                yield article_record(document, article, chapter=chapter, section=section)

In [3]:
records = []
occurrences = Counter()

for path in sorted(SOURCE_DIR.glob('*.json')):
    document = json.loads(path.read_text(encoding='utf-8'))
    for record in iter_articles(document):
        base_id = record['article_id']
        occurrences[base_id] += 1
        if occurrences[base_id] > 1:
            record['id'] = f"{base_id}-{occurrences[base_id]}"
        records.append(record)

with OUTPUT_PATH.open('w', encoding='utf-8') as output:
    for record in records:
        output.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f'Prepared articles: {len(records):,}')
print(f'Output: {OUTPUT_PATH}')

Prepared articles: 2,744
Output: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/search/articles.jsonl


In [4]:
assert records
assert len(records) == len({record['id'] for record in records})
assert all(record['text'].strip() for record in records)
print('Validation passed')
print(json.dumps(records[0], ensure_ascii=False, indent=2))

Validation passed
{
  "id": "1-1",
  "article_id": "1-1",
  "document_id": "1",
  "article_number": 1,
  "text": "មាត្រា ៨ មាត្រា ៤០ មាត្រា ៤១ មាត្រា ៤៧ មាត្រា ៤៨ មាត្រា ៧៥ មាត្រា ៧៧ មាត្រា ៧៨ និងមាត្រា ៨០ នៃច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក ដែលត្រូវប្រកាសឱ្យប្រើដោយព្រះរាជក្រមលេខ នស/រកម/០១១៥/០០១ ចុះថ្ងៃទី ៩ ខែ មករា ឆ្នាំ២០១៥ ត្រូវបានធ្វើវិសោធនកម្មដូចតទៅ",
  "document_title": "ច្បាប់ស្តីពីចរាចរផ្លូវគោក (វិសោធនកម្ម)",
  "document_number": "នស/រកម/០១១៧/០០២",
  "document_type": "Law",
  "source_url": "https://library.ncdd.gov.kh/detail/1",
  "pdf_url": "https://library.ncdd.gov.kh/detail/1/download"
}
